# Modelo A: BiLSTM encoder-decoder con atención

Encoder BiLSTM y decoder LSTM con atención de producto punto, entrenado desde cero con teacher forcing.
Usar GPU (L4 o T4).

In [ ]:
REPO_URL = 'https://github.com/EmilianoRL/NLP_Proyecto.git'
REPO_DIR = 'NLP_Proyecto'

import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q evaluate sacrebleu
if IN_COLAB and not os.path.exists('common'):
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    %cd {REPO_DIR}
elif not os.path.exists('common') and os.path.exists('../common'):
    %cd ..

In [ ]:
import random
import time
import math
import numpy as np
import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

use_gpu = True
device = torch.device('cuda' if use_gpu and torch.cuda.is_available() else 'cpu')

seed = 1234
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
print(f'device: {device.type}')

Datos y vocabularios (los genera el notebook 01):

In [ ]:
from common import checkpoint as ckpt
from common.data import get_splits
from common.vocab import load_vocab, invert, sentence_to_ids, ids_to_sentence
from common.preprocess import clean
from common.datasets import pairs_to_ids, MyDataset, collate_fn

SAVE_DIR = ckpt.setup_save_dir()
DATA_DIR = os.path.join(SAVE_DIR, 'data')
FIG_DIR = os.path.join(SAVE_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

train_pairs, val_pairs, test_pairs = get_splits(DATA_DIR)
eng_word2idx, spa_word2idx, _ = load_vocab(os.path.join(DATA_DIR, 'vocab.json'))
eng_idx2word, spa_idx2word = invert(eng_word2idx), invert(spa_word2idx)
SOS_IDX, EOS_IDX = spa_word2idx['<sos>'], spa_word2idx['<eos>']
print('vocab en:', len(eng_word2idx), '| vocab es:', len(spa_word2idx))

In [ ]:
batch_size = 64

train_src, train_tgt = pairs_to_ids(train_pairs, eng_word2idx, spa_word2idx)
val_src, val_tgt = pairs_to_ids(val_pairs, eng_word2idx, spa_word2idx)

train_dl = DataLoader(MyDataset(train_src, train_tgt), batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_dl = DataLoader(MyDataset(val_src, val_tgt), batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

## Modelo

El código está en `common/model_a.py`. En cada paso del decoder:
score = h<sup>d</sup><sub>i-1</sub> · h<sup>e</sup><sub>j</sub>, α = softmax(score), c<sub>i</sub> = Σ α<sub>ij</sub> h<sup>e</sup><sub>j</sub>,
y la LSTM recibe el embedding de la palabra anterior junto con c<sub>i</sub>.
Como el encoder es bidireccional, el estado del decoder mide 2 × hidden_size.

In [ ]:
from common.model_a import Seq2SeqAttention

# hiperparámetros
emb_dim = 256
hidden_size = 256
dropout = 0.3
lr = 1e-3
n_epochs = 20
patience = 3          # early stopping con val_loss
clip = 1.0            # recorte de gradiente para la LSTM

model = Seq2SeqAttention(len(eng_word2idx), len(spa_word2idx), emb_dim, hidden_size, dropout).to(device)
loss_func = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=lr)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'parámetros entrenables: {n_params:,}')

## Entrenamiento

Cada época se guarda `modeloA_last.pt` en Drive, y `modeloA.pt` cuando mejora la pérdida de validación.
Si se corta la sesión, al volver a correr esta celda sigue desde la última época.

In [ ]:
def evaluate_loss(model, dl):
    # pérdida promedio por token (sin contar padding)
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for eng_batch, spa_batch, lengths in dl:
            eng_batch, spa_batch = eng_batch.to(device), spa_batch.to(device)
            target_input, target_output = spa_batch[:, :-1], spa_batch[:, 1:]
            logits, _ = model(eng_batch, lengths, target_input)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), target_output.reshape(-1),
                                   ignore_index=0, reduction='sum')
            total_loss += loss.item()
            total_tokens += (target_output != 0).sum().item()
    return total_loss / total_tokens

In [ ]:
MODEL_NAME = 'modeloA'

start_epoch, history, extra = ckpt.resume_experiment(model, optimizer, MODEL_NAME, device=device)
history.setdefault('epoch_time', [])
best_val = extra.get('best_val', float('inf'))
bad_epochs = extra.get('bad_epochs', 0)

for epoch in range(start_epoch, n_epochs):
    if bad_epochs >= patience:
        print('early stopping')
        break
    model.train()
    total_loss, total_tokens = 0.0, 0
    start = time.time()
    for eng_batch, spa_batch, lengths in tqdm(train_dl, desc=f'epoch {epoch+1}'):
        eng_batch, spa_batch = eng_batch.to(device), spa_batch.to(device)
        target_input, target_output = spa_batch[:, :-1], spa_batch[:, 1:]
        model.zero_grad()
        logits, _ = model(eng_batch, lengths, target_input)
        loss = loss_func(logits.reshape(-1, logits.size(-1)), target_output.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        n_tokens = (target_output != 0).sum().item()
        total_loss += loss.item() * n_tokens
        total_tokens += n_tokens
    epoch_time = time.time() - start

    val_loss = evaluate_loss(model, val_dl)
    history['train_loss'].append(total_loss / total_tokens)
    history['val_loss'].append(val_loss)
    history['epoch_time'].append(epoch_time)
    print(f'epoch {epoch+1}: train {history["train_loss"][-1]:.4f} | val {val_loss:.4f} | {epoch_time:.0f} s')

    if val_loss < best_val:
        best_val, bad_epochs = val_loss, 0
        ckpt.save_experiment(model, optimizer, history, MODEL_NAME, epoch=epoch)
    else:
        bad_epochs += 1
    ckpt.save_experiment(model, optimizer, history, MODEL_NAME + '_last', epoch=epoch,
                         extra={'best_val': best_val, 'bad_epochs': bad_epochs})

In [ ]:
x = np.arange(len(history['train_loss'])) + 1
plt.plot(x, history['train_loss'])
plt.plot(x, history['val_loss'])
plt.legend(['train', 'val'])
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Modelo A')
plt.grid(True)
plt.savefig(os.path.join(FIG_DIR, 'curvas_modeloA.png'), dpi=150)
plt.show()

## Evaluación en test

Se carga la mejor época según validación. Test solo se usa aquí.

In [ ]:
model, history = ckpt.load_experiment(model, MODEL_NAME, device=device)

test_src, test_tgt = pairs_to_ids(test_pairs, eng_word2idx, spa_word2idx)
test_dl = DataLoader(MyDataset(test_src, test_tgt), batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

test_loss = evaluate_loss(model, test_dl)
print(f'test loss: {test_loss:.4f} | perplejidad: {math.exp(test_loss):.2f}')

In [ ]:
from common.metrics import bleu_report

predictions = []
for eng_batch, spa_batch, lengths in tqdm(test_dl):
    ids, _ = model.translate(eng_batch.to(device), lengths, SOS_IDX, EOS_IDX)
    predictions += [ids_to_sentence(s, spa_idx2word) for s in ids.tolist()]

sources = [en for en, es in test_pairs]
references = [es for en, es in test_pairs]
bleu = bleu_report(sources, predictions, references)
bleu

In [ ]:
from common.metrics import measure_latency, gpu_name, save_results

def translate_one(sentence):
    src = torch.tensor([sentence_to_ids(clean(sentence), eng_word2idx)], device=device)
    ids, alphas = model.translate(src, [src.size(1)], SOS_IDX, EOS_IDX)
    return ids_to_sentence(ids[0].tolist(), spa_idx2word), alphas[0]

latency = measure_latency(translate_one, sources[:300], device)
print(f'latencia: {latency:.1f} ms por oración ({gpu_name()})')

results = {
    'modelo': 'A',
    'parametros': n_params,
    'tiempo_epoca_s': float(np.mean(history['epoch_time'])),
    'epocas': len(history['train_loss']),
    'gpu': gpu_name(),
    'test_loss': test_loss,
    'test_ppl': math.exp(test_loss),
    'bleu_global': bleu['global'],
    'bleu_media': bleu['media'],
    'bleu_larga': bleu['larga'],
    'latencia_ms': latency,
    'predicciones': predictions,
}
save_results(SAVE_DIR, MODEL_NAME, results)

## Ejemplos y mapas de atención

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 100)

# 4 oraciones al azar y una de más de 20 palabras
rng = random.Random(0)
idx = rng.sample(range(len(test_pairs)), 4)
idx.append(next(i for i, en in enumerate(sources) if len(en.split()) > 20))
pd.DataFrame({'inglés': [sources[i] for i in idx],
              'referencia': [references[i] for i in idx],
              'modelo A': [predictions[i] for i in idx]})

In [ ]:
def plot_attention(sentence, filename=None):
    translation, alphas = translate_one(sentence)
    src_tokens = ['<sos>'] + clean(sentence).split() + ['<eos>']
    out_tokens = translation.split() + ['<eos>']
    alphas = alphas[:len(out_tokens)].cpu().numpy()

    fig, ax = plt.subplots(figsize=(0.5 * len(src_tokens) + 2, 0.5 * len(out_tokens) + 2))
    ax.imshow(alphas, cmap='Blues')
    ax.set_xticks(range(len(src_tokens)))
    ax.set_xticklabels(src_tokens, rotation=90)
    ax.set_yticks(range(len(out_tokens)))
    ax.set_yticklabels(out_tokens)
    ax.set_xlabel('inglés')
    ax.set_ylabel('traducción')
    plt.tight_layout()
    if filename:
        plt.savefig(os.path.join(FIG_DIR, filename), dpi=150)
    plt.show()

plot_attention(sources[idx[0]], 'atencion_modeloA_1.png')
plot_attention(sources[idx[-1]], 'atencion_modeloA_2.png')